# Flip scenarios: minimum votes to change a presidential election, 1864-2024

For every U.S. presidential election in `presidential_margins.csv`, this notebook computes the
smallest number of popular votes that would need to move (in the cheapest combination of
states) to:

- **Flip winner** — make the national runner-up reach an electoral college majority, or
- **Break majority** — take the actual winner *below* a majority (so no one reaches 270-style
  threshold), or
- **Tie** — split the electoral college exactly in half (only possible in years with an even
  total EV count).

This is a direct, self-contained port of the algorithm powering the "flip scenario" tool on the
margin-matters site (`build_flip_results.py` / `docs/utils/flipScenarios.js`) — a real 0/1
knapsack, not a greedy heuristic. The only input file needed is `presidential_margins.csv`
sitting next to this notebook.

In [1]:
import math
from collections import defaultdict

import pandas as pd
from IPython.display import Markdown, display

## Load data

In [2]:
df = pd.read_csv('presidential_margins.csv')
print(f"{df.shape[0]:,} rows, years {df['year'].min()}-{df['year'].max()}")
df.head()

2,013 rows, years 1864-2024


,year,abbr,D_votes,R_votes,electoral_votes,T_votes,D_share,R_share,D_candidate,R_candidate,...,third_party_relative_share_str,two_party_margin_str,two_party_margin_delta_str,two_party_national_margin_str,two_party_national_margin_delta_str,two_party_relative_margin_str,two_party_relative_margin_delta_str,elasticity_str,special_case_notes,source_url
0,1864,CA,43838,62140,5,0,0.413621,0.586304,George B. McClellan,Abraham Lincoln,...,-0.01%,R+17.3,0.0,R+10.1,0.0,R+7.2,0.0,0.0,NaN,https://en.wikipedia.org/wiki/1864_United_Stat...
1,1864,CT,42288,44693,6,0,0.486175,0.513825,George B. McClellan,Abraham Lincoln,...,-0.02%,R+2.8,0.0,R+10.1,0.0,D+7.3,0.0,0.0,NaN,https://en.wikipedia.org/wiki/1864_United_Stat...
2,1864,DE,8767,8155,3,0,0.518083,0.481917,George B. McClellan,Abraham Lincoln,...,-0.02%,D+3.6,0.0,R+10.1,0.0,D+13.7,0.0,0.0,NaN,https://en.wikipedia.org/wiki/1864_United_Stat...
3,1864,IA,49525,88500,8,0,0.358812,0.641188,George B. McClellan,Abraham Lincoln,...,-0.02%,R+28.2,0.0,R+10.1,0.0,R+18.2,0.0,0.0,NaN,https://en.wikipedia.org/wiki/1864_United_Stat...
4,1864,IL,158724,189512,16,0,0.455794,0.544206,George B. McClellan,Abraham Lincoln,...,-0.02%,R+8.8,0.0,R+10.1,0.0,D+1.2,0.0,0.0,NaN,https://en.wikipedia.org/wiki/1864_United_Stat...


## Step 1: Load per-unit rows

Each row of `presidential_margins.csv` is one voting unit (a state, DC, or a Maine/Nebraska
congressional district) for one year — plus a `NATIONAL` pseudo-row per year with
`electoral_votes = 0` (it can't be flipped, but it carries the national vote total). We pull out
the D/R/T (top third-party) vote counts and derive each unit's winner, breaking ties D > R > T.

In [3]:
def load_rows(df: pd.DataFrame):
    rows = []
    for r in df.to_dict('records'):
        def num(k, default=0):
            v = r.get(k)
            if v is None or (isinstance(v, float) and pd.isna(v)) or v == '':
                return default
            try:
                return float(v)
            except Exception:
                return default

        row = {
            'year': int(float(r['year'])),
            'abbr': r['abbr'],
            'D_votes': int(num('D_votes', 0)),
            'R_votes': int(num('R_votes', 0)),
            'T_votes': int(num('T_votes', 0)),
            'total_votes': int(num('total_votes', 0)),
            'electoral_votes': int(num('electoral_votes', 0)),
        }
        d, r_, t = row['D_votes'], row['R_votes'], row['T_votes']
        if d >= r_ and d >= t:
            row['party_win'] = 'D'
            row['winner_votes'] = d
            row['runner_up_votes'] = max(r_, t)
        elif r_ >= d and r_ >= t:
            row['party_win'] = 'R'
            row['winner_votes'] = r_
            row['runner_up_votes'] = max(d, t)
        else:
            row['party_win'] = 'T'
            row['winner_votes'] = t
            row['runner_up_votes'] = max(d, r_)
        rows.append(row)
    return rows


def group_by_year(rows):
    by = defaultdict(list)
    for r in rows:
        by[r['year']].append(r)
    return dict(by)

## Step 2: 0/1 knapsack DP

`compute_knapsack` finds the cheapest subset of units whose electoral votes sum to **at least**
a target (used for "flip winner" / "break majority" — overshooting EVs is fine if it's cheaper).
`compute_knapsack_exact` requires **exactly** the target EV count (used for "tie"). Both are a
straightforward `dp[i][v]` = minimum cost to reach exactly `v` EVs using the first `i` units —
exhaustive and optimal, not a greedy pass. With at most ~55 units and a few hundred possible EV
totals per year, the DP table is tiny and this runs instantly even in pure Python.

In [4]:
def compute_knapsack(units, target_ev, cost_func=None):
    """units: list of dicts with keys {abbr, ev, votes_needed}. Returns (chosen_units, min_cost, achieved_ev)."""
    if target_ev <= 0:
        return [], 0, 0
    n = len(units)
    if n == 0 or target_ev > sum(u['ev'] for u in units):
        return [], math.inf, 0

    if cost_func is None:
        cost_func = lambda u: int(u['votes_needed'])
    unit_costs = {id(u): int(cost_func(u)) for u in units}

    # Sort for deterministic reconstruction only -- the DP itself is exhaustive/exact.
    units_sorted = sorted(units, key=lambda u: ((unit_costs[id(u)] / max(1, u['ev'])), u['abbr']))

    INF = 10 ** 18
    max_ev = sum(u['ev'] for u in units_sorted)
    dp = [[INF] * (max_ev + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(1, n + 1):
        u = units_sorted[i - 1]
        ev, votes = u['ev'], unit_costs[id(u)]
        for v in range(max_ev + 1):
            dp[i][v] = dp[i - 1][v]
            if v >= ev and dp[i - 1][v - ev] != INF:
                dp[i][v] = min(dp[i][v], dp[i - 1][v - ev] + votes)

    best_v, best_cost = 0, INF
    for v in range(target_ev, max_ev + 1):
        if dp[n][v] < best_cost:
            best_cost, best_v = dp[n][v], v
    if best_cost >= INF:
        return [], math.inf, 0

    chosen, i, v = [], n, best_v
    while i > 0 and v > 0:
        if dp[i][v] != dp[i - 1][v]:
            u = units_sorted[i - 1]
            chosen.append(u)
            v -= u['ev']
        i -= 1
    return chosen, best_cost, best_v


def compute_knapsack_exact(units, target_ev, cost_func=None):
    """Like compute_knapsack but requires exactly target_ev EVs (used for 'tie')."""
    if target_ev <= 0:
        return [], 0, 0
    n = len(units)
    if n == 0 or target_ev > sum(u['ev'] for u in units):
        return [], math.inf, 0

    INF = 10 ** 18
    max_ev = sum(u['ev'] for u in units)
    if cost_func is None:
        cost_func = lambda u: int(u['votes_needed'])
    unit_costs = {id(u): int(cost_func(u)) for u in units}
    units_sorted = sorted(units, key=lambda u: ((unit_costs[id(u)] / max(1, u['ev'])), u['abbr']))

    dp = [[INF] * (max_ev + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(1, n + 1):
        u = units_sorted[i - 1]
        ev, votes = u['ev'], unit_costs[id(u)]
        for v in range(max_ev + 1):
            dp[i][v] = dp[i - 1][v]
            if v >= ev and dp[i - 1][v - ev] != INF:
                dp[i][v] = min(dp[i][v], dp[i - 1][v - ev] + votes)

    if dp[n][target_ev] >= INF:
        return [], math.inf, 0
    chosen, i, v = [], n, target_ev
    while i > 0 and v > 0:
        if dp[i][v] != dp[i - 1][v]:
            u = units_sorted[i - 1]
            chosen.append(u)
            v -= u['ev']
        i -= 1
    return chosen, dp[n][target_ev], target_ev

## Step 3: Per-year scenario analysis

For a year's rows: compute each party's electoral vote total, the majority threshold
`need = total_ev // 2 + 1`, and the winner/runner-up. Then run the knapsack three ways:

- **classic** ("Flip winner"): give the runner-up `need - runner_ev` more EVs, drawn from units
  the runner-up doesn't already hold. Per-unit cost is the votes needed to make the runner-up the
  plurality winner there: `(winner_votes - runner_votes) // 2 + 1` (bumped up if needed so the
  runner-up also clears the third-party vote count in a 3-way race).
- **no_majority** ("Break majority"): take `winner_ev - (need - 1)` EVs away from the actual
  winner, drawn only from units the winner currently holds. Per-unit cost is the votes needed to
  push the winner below *whichever* party is second there (not necessarily the national runner-up).
- **tie**: only possible when the total EV count is even; an exact-knapsack version of classic.

A handful of 19th/20th-century years have data quirks that make certain units un-flippable as a
matter of historical record (no popular-vote returns, or electoral votes split/awarded to a
third-party slate) -- those are carried over from the original site logic below.

In [5]:
def analyze_year(rows_for_year, metric: str = 'votes'):
    ev_by_party = defaultdict(int)
    total_ev = 0
    year = rows_for_year[0]['year'] if rows_for_year else 0

    for r in rows_for_year:
        ev = int(r['electoral_votes'] or 0)
        if year == 1876 and r['abbr'] == 'CO':
            # No popular-vote returns; fixed 3 EV awarded to R (Hayes), unflippable.
            ev = 3
            total_ev += ev
            ev_by_party['R'] += ev
            continue
        total_ev += ev
        if year == 1960 and r['abbr'] == 'AL' and r['party_win'] in ('D', 'T'):
            ev_by_party['D'] += 5
            ev_by_party['T'] += 6
        elif year == 1948 and r['abbr'] == 'AL' and r['party_win'] in ('D', 'T'):
            ev_by_party['T'] += 11
        else:
            ev_by_party[r['party_win']] += ev

    need = total_ev // 2 + 1
    winner_party = max(ev_by_party.items(), key=lambda kv: kv[1])[0]
    winner_ev = ev_by_party[winner_party]
    others = {p: v for p, v in ev_by_party.items() if p != winner_party}
    if others:
        runner_party = max(others.items(), key=lambda kv: kv[1])[0]
        runner_ev = others[runner_party]
    else:
        runner_party, runner_ev = ('D' if winner_party != 'D' else 'R'), 0

    units = []
    for r in rows_for_year:
        if year == 1876 and r['abbr'] == 'CO':
            continue
        if year == 1868 and r['abbr'] == 'FL':
            continue  # No popular-vote returns; fixed EVs awarded to R (Grant).
        if year == 1864 and r['abbr'] == 'LA':
            continue  # No popular-vote returns; fixed EVs awarded to R (Lincoln).
        if year == 1960 and r['abbr'] == 'AL':
            continue  # Split 5 D + 6 Other, not a single flippable unit.
        if year == 1948 and r['abbr'] == 'AL':
            continue  # All 11 EV to Dixiecrats; D_votes here duplicates T_votes.
        if r['party_win'] == runner_party:
            continue
        ev = int(r['electoral_votes'] or 0)
        if ev <= 0:
            continue

        party_votes = {'D': r['D_votes'], 'R': r['R_votes'], 'T': r['T_votes']}
        winner_votes = r['winner_votes']
        runner_votes = party_votes.get(runner_party, 0)
        votes_to_runner = max(0, (winner_votes - runner_votes) // 2 + 1)
        # In a 3-way race, make sure the runner also clears the third-party count.
        highest_T_competitor = party_votes.get('T', 0) if runner_party != 'T' else 0
        votes_to_runner = max(votes_to_runner, max(0, highest_T_competitor - runner_votes + 1))

        other_parties = [p for p in party_votes if p != r['party_win']]
        best_other_votes = max(party_votes[p] for p in other_parties) if other_parties else 0
        to_party_other = max(other_parties, key=lambda p: party_votes[p]) if other_parties else None
        votes_to_best_other = max(0, (winner_votes - best_other_votes) // 2 + 1)

        units.append({
            'year': r['year'], 'abbr': r['abbr'], 'ev': ev, 'total_votes': int(r['total_votes'] or 0),
            'from_party': r['party_win'],
            'votes_to_runner': votes_to_runner,
            'votes_to_best_other': votes_to_best_other,
            'to_party_other': to_party_other,
        })

    if metric == 'margin':
        # Cost in thousandths of a percent of that unit's turnout, instead of raw votes.
        def cost_func(u):
            tv = max(1, int(u['total_votes'] or 0))
            return int(round(100000 * (int(u['votes_needed']) / tv)))
    else:
        def cost_func(u):
            return int(u['votes_needed'])

    # classic: give the runner-up a majority.
    target_ev_classic = max(0, need - runner_ev)
    units_classic = [
        {**u, 'votes_needed': max(0, u['votes_to_runner']), 'target_party': runner_party}
        for u in units
    ]
    chosen_c, cost_c, ev_c = compute_knapsack(units_classic, target_ev_classic, cost_func)
    if ev_c and (runner_ev + ev_c) < need:
        chosen_c, cost_c, ev_c = [], math.inf, 0

    # no_majority: take the winner below a majority.
    target_away = max(0, winner_ev - (need - 1))
    units_from_winner = [
        {**u, 'votes_needed': max(0, u['votes_to_best_other']), 'target_party': u['to_party_other']}
        for u in units if u['from_party'] == winner_party
    ]
    chosen_n, cost_n, ev_n = compute_knapsack(units_from_winner, target_away, cost_func)
    if ev_n and (winner_ev - ev_n) > (need - 1):
        chosen_n, cost_n, ev_n = [], math.inf, 0

    # tie: exact split, only possible with an even total EV count.
    tie_result = ([], math.inf, 0)
    if total_ev % 2 == 0:
        target_ev_tie = total_ev // 2 - runner_ev
        if target_ev_tie > 0:
            units_for_tie = [
                {**u, 'votes_needed': max(0, u['votes_to_runner']), 'target_party': runner_party}
                for u in units
            ]
            tie_result = compute_knapsack_exact(units_for_tie, target_ev_tie, cost_func)
            if tie_result[2] != 0 and tie_result[2] != target_ev_tie:
                tie_result = ([], math.inf, 0)
    chosen_t, cost_t, ev_t = tie_result

    return {
        'winner_party': winner_party, 'winner_ev': winner_ev,
        'runner_party': runner_party, 'runner_ev': runner_ev, 'need': need,
        'classic': {'cost': int(cost_c if math.isfinite(cost_c) else -1), 'ev': ev_c, 'units': chosen_c},
        'no_majority': {'cost': int(cost_n if math.isfinite(cost_n) else -1), 'ev': ev_n, 'units': chosen_n},
        'tie': {'cost': int(cost_t if math.isfinite(cost_t) else -1), 'ev': ev_t, 'units': chosen_t},
        'total_ev': total_ev, 'metric': metric, 'ev_by_party': dict(ev_by_party),
    }

In [6]:
rows = load_rows(df)
by_year = group_by_year(rows)

results = {}
for year in sorted(by_year.keys()):
    for metric in ('votes', 'margin'):
        results[(year, metric)] = analyze_year(by_year[year], metric=metric)

print(f"Computed scenarios for {len({y for y, m in results})} election years.")

Computed scenarios for 41 election years.


## Step 4: Render a scenario as a markdown table

`render_flip_scenario(year, flip_type, metric='votes')` reproduces the site's "Applied flips"
table: the summary badges (votes changed, % of national vote, states flipped, resulting EC
split) followed by a per-state before/after table. `flip_type` accepts `'classic'`/`'winner'`,
`'no_majority'`/`'majority'`, or `'tie'`.

In [7]:
FLIP_ALIASES = {
    'classic': 'classic', 'winner': 'classic', 'flip_winner': 'classic',
    'no_majority': 'no_majority', 'majority': 'no_majority', 'break_majority': 'no_majority',
    'tie': 'tie',
}


def format_pct(pct):
    if not math.isfinite(pct):
        return '0%'
    if abs(pct) < 0.01:
        return f'{pct:.2e}%'
    return f'{pct:.4f}%'


def render_flip_scenario(year, flip_type, metric='votes', show=True):
    mode = FLIP_ALIASES.get(str(flip_type).lower())
    if mode is None:
        raise ValueError(f"Unknown flip_type {flip_type!r}; expected one of {sorted(set(FLIP_ALIASES))}")
    key = (year, metric)
    if key not in results:
        raise ValueError(f"No data for year={year}, metric={metric!r}")
    res = results[key]
    mode_res = res[mode]
    label = {'classic': 'flip the winner', 'no_majority': 'break the majority', 'tie': 'force a tie'}[mode]

    if mode_res['cost'] < 0:
        msg = f"No feasible scenario to {label} in {year} (metric={metric})."
        if show:
            display(Markdown(f"*{msg}*"))
        return msg

    rows_for_year = by_year[year]
    row_by_abbr = {r['abbr']: r for r in rows_for_year}
    metric_label = 'min margin' if metric == 'margin' else 'min votes'
    mode_label = {'classic': 'Flip winner', 'no_majority': 'Break majority', 'tie': 'Tie'}[mode]

    votes_sum = sum(u['votes_needed'] for u in mode_res['units'])
    states_flipped = len(mode_res['units'])
    nat_row = row_by_abbr.get('NATIONAL')
    total_national_votes = nat_row['total_votes'] if nat_row else 0
    pct_of_total = (votes_sum / total_national_votes * 100) if total_national_votes else 0.0

    pre_ev = dict(res['ev_by_party'])
    # Post-flip EC split by party, using each chosen unit's actual destination party.
    post_ev = dict(pre_ev)
    for u in mode_res['units']:
        post_ev[u['from_party']] = post_ev.get(u['from_party'], 0) - u['ev']
        to_p = u.get('target_party')
        if to_p:
            post_ev[to_p] = post_ev.get(to_p, 0) + u['ev']
    if post_ev.get('T', 0) != 0:
        ec_line1 = f"EC D {pre_ev.get('D', 0)} | O {pre_ev.get('T', 0)} | R {pre_ev.get('R', 0)}"
        ec_line2 = f"EC D {post_ev.get('D', 0)} | O {post_ev.get('T', 0)} | R {post_ev.get('R', 0)}"
    else:
        ec_line1 = f"EC D {pre_ev.get('D', 0)} | R {pre_ev.get('R', 0)}"
        ec_line2 = f"EC D {post_ev.get('D', 0)} | R {post_ev.get('R', 0)}"
    ec_line = f"{ec_line1} → {ec_line2}"
    lines = [
        f"**{year} — {mode_label}** (optimize: {metric_label})",
        '',
        f"VOTES CHANGED: **{votes_sum:,}**  |  % OF TOTAL: **{format_pct(pct_of_total)}**  |  "
        f"STATES FLIPPED: **{states_flipped}**  ", 
        f"{ec_line}",
        '',
    ]
    if not mode_res['units']:
        lines.append(f"*Already true — no votes need to change to {label} in {year}.*")
    else:
        lines += [
            f"*Applied flips (optimize: {metric_label}):*",
            '',
            '| State | EV | D | R | Δ votes |',
            '|---|---|---|---|---|',
        ]
        for u in sorted(mode_res['units'], key=lambda u: u['votes_needed']):
            row = row_by_abbr.get(u['abbr'])
            if row is None:
                continue
            d0, r0 = row['D_votes'], row['R_votes']
            vt = max(0, int(u['votes_needed']))
            if d0 >= r0:
                d1, r1 = max(0, d0 - vt), r0 + vt
            else:
                d1, r1 = d0 + vt, max(0, r0 - vt)
            pct_state = round(100.0 * vt / row['total_votes'], 3) if row['total_votes'] else 0.0
            lines.append(
                f"| {u['abbr']} | {u['ev']} | {d0:,}<br>→ {d1:,} | {r0:,}<br>→ {r1:,} | "
                f"{vt:,} ({pct_state}%) |"
            )

    md_text = '\n'.join(lines)
    if show:
        display(Markdown(md_text))
    return md_text

### Example usage

In [8]:
_ = render_flip_scenario(2000, 'winner')
_ = render_flip_scenario(2016, 'winner')
_ = render_flip_scenario(2020, 'break_majority')
_ = render_flip_scenario(2024, 'winner')

**2000 — Flip winner** (optimize: min votes)

VOTES CHANGED: **269**  |  % OF TOTAL: **2.55e-04%**  |  STATES FLIPPED: **1**  
EC D 267 | R 271 → EC D 292 | R 246

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| FL | 25 | 2,912,253<br>→ 2,912,522 | 2,912,790<br>→ 2,912,521 | 269 (0.005%) |

**2016 — Flip winner** (optimize: min votes)

VOTES CHANGED: **38,875**  |  % OF TOTAL: **0.0284%**  |  STATES FLIPPED: **3**  
EC D 232 | R 306 → EC D 278 | R 260

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| MI | 16 | 2,268,839<br>→ 2,274,192 | 2,279,543<br>→ 2,274,190 | 5,353 (0.112%) |
| WI | 10 | 1,382,536<br>→ 1,393,911 | 1,405,284<br>→ 1,393,909 | 11,375 (0.382%) |
| PA | 20 | 2,926,441<br>→ 2,948,588 | 2,970,733<br>→ 2,948,586 | 22,147 (0.359%) |

**2020 — Break majority** (optimize: min votes)

VOTES CHANGED: **21,461**  |  % OF TOTAL: **0.0135%**  |  STATES FLIPPED: **3**  
EC D 306 | R 232 → EC D 269 | R 269

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| AZ | 11 | 1,672,143<br>→ 1,666,914 | 1,661,686<br>→ 1,666,915 | 5,229 (0.154%) |
| GA | 16 | 2,473,633<br>→ 2,467,743 | 2,461,854<br>→ 2,467,744 | 5,890 (0.118%) |
| WI | 10 | 1,630,866<br>→ 1,620,524 | 1,610,184<br>→ 1,620,526 | 10,342 (0.314%) |

**2024 — Flip winner** (optimize: min votes)

VOTES CHANGED: **114,885**  |  % OF TOTAL: **0.0740%**  |  STATES FLIPPED: **3**  
EC D 226 | R 312 → EC D 270 | R 268

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| WI | 10 | 1,668,229<br>→ 1,682,928 | 1,697,626<br>→ 1,682,927 | 14,699 (0.429%) |
| MI | 15 | 2,736,533<br>→ 2,776,585 | 2,816,636<br>→ 2,776,584 | 40,052 (0.707%) |
| PA | 19 | 3,423,042<br>→ 3,483,176 | 3,543,308<br>→ 3,483,174 | 60,134 (0.852%) |

## Step 5: Top closest elections by cost to flip the winner

The elections (1864-2024) where the fewest raw popular votes, moved in the cheapest
combination of states, would have flipped the winner of the electoral college.

In [9]:
# Number of elections to display and the column used to rank them.
N = 10
metric = r'% of national vote'

summary_records = []
for year in sorted(by_year.keys()):
    res = results[(year, 'votes')]
    cost = res['classic']['cost']
    n_states = len(res['classic']['units'])

    # Get the national vote total and candidate names for this year.
    row_by_abbr = {r['abbr']: r for r in by_year[year]}
    nat = row_by_abbr.get('NATIONAL')
    total_votes = nat['total_votes'] if nat else 0
    year_df = df[df['year'] == year]

    # Store the summary row used to rank elections.
    summary_records.append({
        'year': year,
        'Democrat': year_df['D_candidate'].iloc[0],
        'Republican': year_df['R_candidate'].iloc[0],
        'Votes to flip': cost,
        r'# of Flipped states': n_states,
        r'% of national vote': f'{(cost / total_votes) if total_votes else 0.0:.4%}',
    })

# Rank the closest elections and use one-based row labels for display.
topNflip = pd.DataFrame(summary_records).sort_values(metric).head(N).reset_index(drop=True)
# display the percent of national vote in scientific notation
#topNflip[r'% of national vote'] = topNflip[r'% of national vote'].apply(lambda x: f"{x:.2e}")
topNflip.index += 1

title = f"Top {N} closest elections electorally from 1864-2024 by {metric.replace('_', ' ')}"
display(Markdown(f"## {title}"))
topNflip

## Top 10 closest elections electorally from 1864-2024 by % of national vote

,year,Democrat,Republican,Votes to flip,# of Flipped states,% of national vote
1,2000,Al Gore,George W. Bush,269,1,0.0003%
2,1876,Samuel J. Tilden,Rutherford B. Hayes,445,1,0.0053%
3,1884,Grover Cleveland,James G. Blaine,575,1,0.0057%
4,1916,Woodrow Wilson,Charles Evans Hughes,1887,1,0.0102%
5,1976,Jimmy Carter,Gerald Ford,9246,2,0.0113%
6,1960,John F. Kennedy,Richard Nixon,11874,5,0.0173%
7,2020,Joe Biden,Donald Trump,32507,4,0.0205%
8,2016,Hillary Clinton,Donald Trump,38875,3,0.0284%
9,2004,John Kerry,George W. Bush,46368,4,0.0379%
10,1948,Harry S. Truman,Thomas E. Dewey,29294,3,0.0600%


## Step 6: Top closest elections by fragility

The elections (1864-2024) where the fewest raw popular votes, moved in the cheapest
combination of states, would have flipped the winner or broken the electoral college majority —
whichever of the two is cheaper for that year.

In [10]:
# Find the cheaper feasible scenario for each election year.
summary_records = []
for year in sorted(by_year.keys()):
    res = results[(year, 'votes')]
    candidates = []

    # Include each feasible scenario with its raw-vote cost and number of flipped units.
    if res['classic']['cost'] >= 0:
        candidates.append(('classic', res['classic']['cost'], len(res['classic']['units'])))
    if res['no_majority']['cost'] >= 0:
        candidates.append(('no_majority', res['no_majority']['cost'], len(res['no_majority']['units'])))
    if not candidates:
        continue

    # Select the least costly way to change the electoral outcome.
    mode, cost, n_states = min(candidates, key=lambda c: c[1])

    # Get the national vote total and candidate names for this year.
    row_by_abbr = {r['abbr']: r for r in by_year[year]}
    nat = row_by_abbr.get('NATIONAL')
    total_votes = nat['total_votes'] if nat else 0
    year_df = df[df['year'] == year]

    # Store the summary row used to rank elections.
    summary_records.append({
        'year': year,
        'Democrat': year_df['D_candidate'].iloc[0],
        'Republican': year_df['R_candidate'].iloc[0],
        'mode': 'Flip winner' if mode == 'classic' else 'Break majority',
        'Votes to flip': cost,
        r'# of Flipped states': n_states,
        r'% of national vote': round((cost / total_votes * 100) if total_votes else 0.0, 4),
    })

# Rank the closest elections and use one-based row labels for display.
topNfragile = pd.DataFrame(summary_records).sort_values(metric).head(N).reset_index(drop=True)
topNfragile.index += 1

title = f"Top {N} most fragile elections electorally from 1864-2024 by {metric.replace('_', ' ')}"
display(Markdown(f"## {title}"))
topNfragile

## Top 10 most fragile elections electorally from 1864-2024 by % of national vote

,year,Democrat,Republican,mode,Votes to flip,# of Flipped states,% of national vote
1,2000,Al Gore,George W. Bush,Flip winner,269,1,0.0003
2,1876,Samuel J. Tilden,Rutherford B. Hayes,Flip winner,445,1,0.0053
3,1884,Grover Cleveland,James G. Blaine,Flip winner,575,1,0.0057
4,1960,John F. Kennedy,Richard Nixon,Break majority,6883,4,0.0100
5,1916,Woodrow Wilson,Charles Evans Hughes,Flip winner,1887,1,0.0102
6,1976,Jimmy Carter,Gerald Ford,Flip winner,9246,2,0.0113
7,2020,Joe Biden,Donald Trump,Break majority,21461,3,0.0135
8,2004,John Kerry,George W. Bush,Break majority,18776,3,0.0154
9,2016,Hillary Clinton,Donald Trump,Break majority,30768,3,0.0225
10,1948,Harry S. Truman,Thomas E. Dewey,Break majority,12487,2,0.0256


In [11]:
# Print CLI commands to generate a flip-scenario GIF for each row above,
# for tools/generate_flip_gif.mjs. Requires `npm start` running in another
# terminal and ffmpeg on PATH; see that script's header comment.
_MODE_TOKEN = {'Flip winner': 'classic', 'Break majority': 'no_majority'}
year_mode_combos = set()
for _, row in topNfragile.iterrows():
    mode = _MODE_TOKEN[row['mode']]
    year_mode_combos.add((row['year'], mode))
for _, row in topNflip.iterrows():
    year_mode_combos.add((row['year'], 'classic'))
# sort by year
for year, mode in sorted(year_mode_combos):
    print(f"node tools/generate_flip_gif.mjs --year {year} --mode {mode} --metric votes --dim --rendervotes")
# add 2024
print(f"node tools/generate_flip_gif.mjs --year 2024 --mode classic --metric votes --dim --rendervotes")

node tools/generate_flip_gif.mjs --year 1876 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1884 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1916 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1948 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1948 --mode no_majority --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1960 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1960 --mode no_majority --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 1976 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 2000 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 2004 --mode classic --metric votes --dim --rendervotes
node tools/generate_flip_gif.mjs --year 20

## Table Results

In [12]:
# generate tables for each of the top N closest elections in separate cells using render_flip_scenario
for num, (i, row) in enumerate(topNflip.iterrows(), start=1):
    display(Markdown(f"### {num}. {row['year']}"))
    _ = render_flip_scenario(row['year'], 'winner')

### 1. 2000

**2000 — Flip winner** (optimize: min votes)

VOTES CHANGED: **269**  |  % OF TOTAL: **2.55e-04%**  |  STATES FLIPPED: **1**  
EC D 267 | R 271 → EC D 292 | R 246

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| FL | 25 | 2,912,253<br>→ 2,912,522 | 2,912,790<br>→ 2,912,521 | 269 (0.005%) |

### 2. 1876

**1876 — Flip winner** (optimize: min votes)

VOTES CHANGED: **445**  |  % OF TOTAL: **5.29e-03%**  |  STATES FLIPPED: **1**  
EC D 184 | R 185 → EC D 191 | R 178

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| SC | 7 | 90,897<br>→ 91,342 | 91,786<br>→ 91,341 | 445 (0.244%) |

### 3. 1884

...

**2016 — Flip winner** (optimize: min votes)

VOTES CHANGED: **38,875**  |  % OF TOTAL: **0.0284%**  |  STATES FLIPPED: **3**  
EC D 232 | R 306 → EC D 278 | R 260

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| MI | 16 | 2,268,839<br>→ 2,274,192 | 2,279,543<br>→ 2,274,190 | 5,353 (0.112%) |
| WI | 10 | 1,382,536<br>→ 1,393,911 | 1,405,284<br>→ 1,393,909 | 11,375 (0.382%) |
| PA | 20 | 2,926,441<br>→ 2,948,588 | 2,970,733<br>→ 2,948,586 | 22,147 (0.359%) |

### 9. 2004

**2004 — Flip winner** (optimize: min votes)

VOTES CHANGED: **46,368**  |  % OF TOTAL: **0.0379%**  |  STATES FLIPPED: **4**  
EC D 252 | R 286 → EC D 270 | R 268

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| NM | 5 | 370,942<br>→ 373,937 | 376,930<br>→ 373,935 | 2,995 (0.396%) |
| IA | 7 | 741,898<br>→ 746,928 | 751,957<br>→ 746,927 | 5,030 (0.334%) |
| NV | 5 | 397,190<br>→ 407,941 | 418,690<br>→ 407,939 | 10,751 (1.296%) |
| NE-02 | 1 | 97,858<br>→ 125,450 | 153,041<br>→ 125,449 | 27,592 (10.862%) |

### 10. 1948

**1948 — Flip winner** (optimize: min votes)

VOTES CHANGED: **29,294**  |  % OF TOTAL: **0.0600%**  |  STATES FLIPPED: **3**  
EC D 304 | O 38 | R 189 → EC D 226 | O 38 | R 267

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| OH | 25 | 1,452,791<br>→ 1,449,237 | 1,445,684<br>→ 1,449,238 | 3,554 (0.121%) |
| CA | 25 | 1,913,134<br>→ 1,904,201 | 1,895,269<br>→ 1,904,202 | 8,933 (0.222%) |
| IL | 28 | 1,994,715<br>→ 1,977,908 | 1,961,103<br>→ 1,977,910 | 16,807 (0.422%) |

In [13]:
# generate tables for each of the top N closest elections in separate cells using render_flip_scenario
for num, (i, row) in enumerate(topNfragile.iterrows(), start=1):
    display(Markdown(f"### {num}. {row['year']}"))
    _ = render_flip_scenario(row['year'], 'winner' if row['mode'] == 'Flip winner' else 'break_majority')

### 1. 2000

**2000 — Flip winner** (optimize: min votes)

VOTES CHANGED: **269**  |  % OF TOTAL: **2.55e-04%**  |  STATES FLIPPED: **1**  
EC D 267 | R 271 → EC D 292 | R 246

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| FL | 25 | 2,912,253<br>→ 2,912,522 | 2,912,790<br>→ 2,912,521 | 269 (0.005%) |

### 2. 1876

**1876 — Flip winner** (optimize: min votes)

VOTES CHANGED: **445**  |  % OF TOTAL: **5.29e-03%**  |  STATES FLIPPED: **1**  
EC D 184 | R 185 → EC D 191 | R 178

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| SC | 7 | 90,897<br>→ 91,342 | 91,786<br>→ 91,341 | 445 (0.244%) |

### 3. 1884

...

**2004 — Break majority** (optimize: min votes)

VOTES CHANGED: **18,776**  |  % OF TOTAL: **0.0154%**  |  STATES FLIPPED: **3**  
EC D 252 | R 286 → EC D 269 | R 269

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| NM | 5 | 370,942<br>→ 373,937 | 376,930<br>→ 373,935 | 2,995 (0.396%) |
| IA | 7 | 741,898<br>→ 746,928 | 751,957<br>→ 746,927 | 5,030 (0.334%) |
| NV | 5 | 397,190<br>→ 407,941 | 418,690<br>→ 407,939 | 10,751 (1.296%) |

### 9. 2016

**2016 — Break majority** (optimize: min votes)

VOTES CHANGED: **30,768**  |  % OF TOTAL: **0.0225%**  |  STATES FLIPPED: **3**  
EC D 232 | R 306 → EC D 269 | R 269

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| NE-02 | 1 | 131,030<br>→ 134,298 | 137,564<br>→ 134,296 | 3,268 (1.12%) |
| MI | 16 | 2,268,839<br>→ 2,274,192 | 2,279,543<br>→ 2,274,190 | 5,353 (0.112%) |
| PA | 20 | 2,926,441<br>→ 2,948,588 | 2,970,733<br>→ 2,948,586 | 22,147 (0.359%) |

### 10. 1948

**1948 — Break majority** (optimize: min votes)

VOTES CHANGED: **12,487**  |  % OF TOTAL: **0.0256%**  |  STATES FLIPPED: **2**  
EC D 304 | O 38 | R 189 → EC D 254 | O 38 | R 239

*Applied flips (optimize: min votes):*

| State | EV | D | R | Δ votes |
|---|---|---|---|---|
| OH | 25 | 1,452,791<br>→ 1,449,237 | 1,445,684<br>→ 1,449,238 | 3,554 (0.121%) |
| CA | 25 | 1,913,134<br>→ 1,904,201 | 1,895,269<br>→ 1,904,202 | 8,933 (0.222%) |

## Notes / caveats to carry forward

- **Historical exclusions** baked into `analyze_year` (matching the site, for data-quality
  reasons rather than modeling choices): Colorado 1876, Florida 1868, and Louisiana 1864 have no
  popular-vote returns in this dataset and their fixed EVs are excluded from flip consideration;
  Alabama 1960 (5 D + 6 Other split) and Alabama 1948 (all 11 EV to Dixiecrats, where `D_votes`
  is a duplicate of `T_votes`) are excluded as single flippable units.
- **Maine/Nebraska congressional districts** (`ME-01`, `ME-02`, `NE-01`, `NE-02`, `NE-03`, plus
  the statewide `ME-AL`/`NE-AL` rows) are treated as independent flippable units, same as any
  state.
- **`T_votes`** is the *top* third-party candidate's votes only, not the sum of all minor
  candidates (`third_party_votes`) — the knapsack always treats third parties as a single "T"
  slate per unit.
- Two cost metrics are computed for every year: `'votes'` (raw vote count, used above) and
  `'margin'` (cost in thousandths of a percent of that unit's own turnout, which tends to favor
  flipping small, close states over large ones).